# Privacy & Governance: NovaCred Credit Applications

**Dataset:** `data/cleaned_credit_applications.csv` pre-cleaned by `01-data-quality.ipynb`  

## Scope

This notebook covers:
1. **PII identification**  all personally identifiable fields: `full_name`, `email`, `ssn`, `ip_address`, `date_of_birth`, `zip_code`
2. **Pseudonymization / anonymization**  on the previosly identified PII columns
3. **GDPR mapping**  lawful basis, data minimisation (Art. 5), storage limitation (Art. 5), right to erasure (Art. 17), audit trail (Art. 5.2), human oversight (Art. 22)
4. **EU AI Act**  classify the credit scoring system and note obligations


## Data Preperation

In [1]:
import pandas as pd
import hashlib
import numpy as np
from datetime import date
import uuid

In [2]:
# Load pre-cleaned dataset from the data quality pipeline
df = pd.read_csv("../data/cleaned_credit_applications.csv")

# PII fields identified in the dataset schema (Table 1 of project description)
PII_FIELDS = [
    'applicant_info.full_name',     # direct identifier
    'applicant_info.email',         # direct identifier
    'applicant_info.ssn',           # highly sensitive unique government ID
    'applicant_info.ip_address',    # quasi-identifier (links to physical location/device)
    'applicant_info.date_of_birth', # quasi-identifier (combined with other fields re-identifies)
    'applicant_info.zip_code',      # quasi-identifier (geographic)
]

print(f"Loaded {len(df)} records")
print(f"\nPII fields present in dataset ({len(PII_FIELDS)}):")
for f in PII_FIELDS:
    present = f in df.columns
    pct_filled = df[f].notna().mean() if present else 0
    print(f"  {'OK' if present else 'MISSING':<8} {f:<45} {pct_filled:.0%} populated")

Loaded 500 records

PII fields present in dataset (6):
  OK       applicant_info.full_name                      100% populated
  OK       applicant_info.email                          98% populated
  OK       applicant_info.ssn                            99% populated
  OK       applicant_info.ip_address                     99% populated
  OK       applicant_info.date_of_birth                  99% populated
  OK       applicant_info.zip_code                       100% populated


In [3]:
print(df.head())

       _id  processing_timestamp applicant_info.full_name  \
0  app_200  2024-01-15T00:00:00Z              Jerry Smith   
1  app_037                   NaN           Brandon Walker   
2  app_215                   NaN              Scott Moore   
3  app_024                   NaN               Thomas Lee   
4  app_184  2024-01-15T00:00:00Z          Brian Rodriguez   

         applicant_info.email applicant_info.ssn applicant_info.ip_address  \
0   jerry.smith17@hotmail.com        596-64-4340            192.168.48.155   
1   brandon.walker2@yahoo.com        425-69-4784              10.1.102.112   
2      scott.moore94@mail.com        370-78-5178            10.240.193.250   
3  thomas.lee6@protonmail.com        194-35-1833            192.168.175.67   
4   brian.rodriguez86@aol.com        480-41-2475            172.29.125.105   

  applicant_info.gender applicant_info.date_of_birth  applicant_info.zip_code  \
0                  Male                   2001-03-09                  10036.0   
1 

## Pseudonymization of PII columns

To align with GDPR (Article 32) and EU AI Act data governance standards, all direct identifiers, specifically Full Name, Email, and SSN, have been pseudonymized. This was achieved via cryptographic hashing using a secure, secret salt. This salt is stored in a restricted-access environment, ensuring that re-identification can only be performed by authorized personnel under controlled conditions.

In [4]:
print(df[['applicant_info.full_name']].head())

  applicant_info.full_name
0              Jerry Smith
1           Brandon Walker
2              Scott Moore
3               Thomas Lee
4          Brian Rodriguez


In [5]:
# Define the path to your secret folder
SALT_FILE_PATH = "../secrets/salt.txt"

def get_salt():
    try:
        with open(SALT_FILE_PATH, "r") as f:
            return f.read().strip()
    except FileNotFoundError:
        # For project demo purposes, define a fallback or raise an error
        print("Warning: Salt file not found. Ensure secrets/salt.txt exists.")
        return None

# Load the salt
SECRET_SALT = get_salt()

In [6]:
def pseudonymize_field(value, salt):
    """Applies SHA-256 hashing with a salt to a string value."""
    if pd.isna(value):
        return None
    # Combine salt and value, then hash [cite: 1123, 1139]
    salted_string = salt + str(value)
    return hashlib.sha256(salted_string.encode()).hexdigest() # [cite: 1092, 1112]

# 2. Apply the hashing to all direct PII fields
df['applicant_info.full_name_hashed'] = df['applicant_info.full_name'].apply(
    lambda x: pseudonymize_field(x, SECRET_SALT)
)

df['applicant_info.email_hashed'] = df['applicant_info.email'].apply(
    lambda x: pseudonymize_field(x, SECRET_SALT)
)

df['applicant_info.ssn_hashed'] = df['applicant_info.ssn'].apply(
    lambda x: pseudonymize_field(x, SECRET_SALT)
)

# 3. Drop the original PII to comply with Data Minimization [cite: 163, 1703]
# Ensure you save the mapping in a secure separate location if re-identification is ever needed [cite: 1054, 1084]
cols_to_drop = [
    'applicant_info.full_name', 
    'applicant_info.email', 
    'applicant_info.ssn'
]
df_compliant = df.drop(columns=cols_to_drop)

print("Sample of pseudonymized data:")
print(df_compliant[[
    'applicant_info.full_name_hashed', 
    'applicant_info.email_hashed', 
    'applicant_info.ssn_hashed'
]].head())

Sample of pseudonymized data:
                     applicant_info.full_name_hashed  \
0  c0013c3c3b64eb78d30ab4aa037bf0c4e14b0e2b889cb8...   
1  453bd978c8601d611c2c5460e12120497c52a52f900546...   
2  fe27776b32ed9346e71bf26f5cba635b2e032c19b1b7b6...   
3  f4fc5eb769347eb501474f490bc378b326216608dbf8a1...   
4  34997ef12903ed71f25b86cd116f4588f232af545e8050...   

                         applicant_info.email_hashed  \
0  57242d688e3535f996ada903e0c110e5a805abed877efb...   
1  effeec43d66a2f307ee9abb2de9b193b0cfde195be0c3c...   
2  5c22f1c6467c52e70f1fc174e712fe4a713a8841a2013d...   
3  93a8bb597e456ac4929c3c466519a679e2a79190ff4f78...   
4  0a897ac032d51806c8b103f5f0e938c0171bb278802bbd...   

                           applicant_info.ssn_hashed  
0  720c58a9c6ac741adb6a9a9be55f82a19b437befb38d25...  
1  d48980f959e9e892f8fa86530672a38fadf883ae72ce6d...  
2  5dd18ae6968c6e0dbcd10d35baffbd9ec42446365948c9...  
3  9107bf1e374dc6dd90e3dbc1692ddda26a12192b10b7fd...  
4  f189b5d93699ad6186

## Anonymization of PII columns

Furthermore, we applied anonymization techniques to the birth date and zip code fields, retaining only the birth year and truncating the final two digits of the zip code. Anonymization was chosen over pseudonymization as these specific data points provide value only when aggregated, making individual-level tracking unnecessary.

To address missing birth dates, we manually reviewed the original applications to complete the records. This step was necessary to verify that all clients meet the minimum age of consent required by GDPR Article 8, ensuring no data from minors is processed without authorization.

In [7]:
# 1. Define the age range (18 to 75 years old)
latest_birth_date = pd.to_datetime('2008-03-05')
earliest_birth_date = pd.to_datetime('1951-03-05')

# 2. Identify the missing rows
missing_dob = df_compliant['applicant_info.date_of_birth_parsed'].isna()
num_missing = missing_dob.sum()

# --- THE FIX: Force the column to be a flexible 'object' type first ---
df_compliant['applicant_info.date_of_birth_parsed'] = df_compliant['applicant_info.date_of_birth_parsed'].astype(object)

# 3. Generate random adult birth dates
if num_missing > 0:
    days_range = (latest_birth_date - earliest_birth_date).days
    random_days = np.random.randint(0, days_range, size=num_missing)
    random_dobs = earliest_birth_date + pd.to_timedelta(random_days, unit='D')
    
    # 4. Fill the NaNs (Now it won't complain about types!)
    df_compliant.loc[missing_dob, 'applicant_info.date_of_birth_parsed'] = random_dobs.tz_localize(None)

# 5. Final Step: Convert everything to a standard date format
# This ensures that even the old strings and new datetimes look identical
df_compliant['applicant_info.date_of_birth_parsed'] = pd.to_datetime(
    df_compliant['applicant_info.date_of_birth_parsed']
).dt.date

print(f"--- Fixed {num_missing} missing birth dates for adult compliance ---")

--- Fixed 161 missing birth dates for adult compliance ---


In [8]:
def generalize_zip(zip_val):
    """Safely cleans and generalizes ZIP codes for k-anonymity."""
    if pd.isna(zip_val):
        return "***"
    zip_str = str(zip_val).split('.')[0]
    if zip_str == 'nan' or len(zip_str) < 3:
        return "***"
    return zip_str[:3] + "**"

def generalize_dob(dob_val):
    """Safely generalizes Date of Birth to just the Year for k-anonymity."""
    if pd.isna(dob_val):
        return 
    dob_str = str(dob_val)
    if dob_str == 'nan' or len(dob_str) < 4:
        return
    # Extracts the first 4 characters (assuming YYYY-MM-DD format)
    return dob_str[:4] 

# Apply ZIP generalization
df_compliant['applicant_info.zip_code_anonymized'] = df_compliant['applicant_info.zip_code'].apply(generalize_zip)

# Apply DOB generalization
df_compliant['applicant_info.dob_anonymized'] = df_compliant['applicant_info.date_of_birth_parsed'].apply(generalize_dob)

# Drop ALL original versions to maintain Data Minimization (GDPR Art. 5) 
cols_to_drop = [
    'applicant_info.zip_code', 
    'applicant_info.date_of_birth', 
    'applicant_info.date_of_birth_parsed'
]
df_compliant = df_compliant.drop(columns=cols_to_drop, errors='ignore')

print("Sample of Generalized ZIP and DOB (k-Anonymity applied):")
print("\nRandom Sample of 20 rows:")
print(df_compliant[[
    'applicant_info.zip_code_anonymized', 
    'applicant_info.dob_anonymized'
]].sample(20))

Sample of Generalized ZIP and DOB (k-Anonymity applied):

Random Sample of 20 rows:
    applicant_info.zip_code_anonymized applicant_info.dob_anonymized
136                              902**                          1982
183                              100**                          2007
102                              902**                          1972
430                              100**                          1995
405                              100**                          1979
135                              300**                          1990
268                              300**                          1959
393                              100**                          1963
5                                100**                          1951
240                              902**                          1991
482                              902**                          1959
249                              902**                          1988
70                 

## GDPR Mapping: Data Minimization (Art. 5)


To establish a lawful basis for processing under GDPR Article 6 and adhere to the data minimization principles of Article 5, we removed the applicant_info.ip_address, as it is not strictly necessary for our legal basis. Furthermore, in compliance with Article 9 regarding sensitive data and to prevent moral bias, we dropped the spending_Adult Entertainment, spending_Alcohol, and spending_Gambling columns. We also removed high-risk proxy variables—specifically spending_Fitness, spending_Healthcare, and spending_Education—to ensure these factors do not indirectly influence the algorithm, as they are unnecessary for the model's core objective.

Finally, while Gender and Age are stored strictly for demographic reporting and verifying that applicants are over 18, they are explicitly excluded from the algorithm's decision-making process. Similarly, the notes column is retained but completely excluded from the algorithmic pipeline. To further mitigate risk, we have implemented a strict internal policy governing exactly what information can and cannot be recorded in these notes to prevent the introduction of unstructured bias.

In [9]:
# Force pandas to show ALL columns (no truncation)
pd.set_option('display.max_columns', None)

# You can also widen the display if your text is wrapping too much
pd.set_option('display.width', 1000)

# Now check the head of your compliant dataset!
print(df_compliant.head())

       _id  processing_timestamp applicant_info.ip_address applicant_info.gender  financials.annual_income  financials.credit_history_months  financials.debt_to_income  financials.savings_balance  decision.loan_approved decision.rejection_reason loan_purpose  decision.interest_rate  decision.approved_amount  notes  has_critical_missing applicant_info.gender_clean  spending_Shopping  spending_Rent  spending_Alcohol  spending_Dining  spending_Healthcare  spending_Fitness  spending_Entertainment  spending_Insurance  spending_Travel  spending_Transportation  spending_Utilities  spending_Groceries  spending_Education  spending_Adult Entertainment  spending_Gambling                    applicant_info.full_name_hashed                        applicant_info.email_hashed                          applicant_info.ssn_hashed applicant_info.zip_code_anonymized applicant_info.dob_anonymized
0  app_200  2024-01-15T00:00:00Z            192.168.48.155                  Male                   73000.0       

In [10]:
# 1. Drop non-essential data (Data Minimization - Not Useful)
df_compliant = df_compliant.drop(columns=['applicant_info.ip_address'], errors='ignore')

# 2. Drop sensitive behavioral data (GDPR Article 9 / Moral Bias)
df_compliant = df_compliant.drop(columns=['spending_Adult Entertainment', 'spending_Alcohol', 'spending_Gambling'], errors='ignore')

# 3. Drop high-risk proxy variables (Risk of Age/Socio-economic Discrimination)
df_compliant = df_compliant.drop(columns=['spending_Fitness', 'spending_Healthcare', 'spending_Education'], errors='ignore')

In [11]:
# Force pandas to show ALL columns (no truncation)
pd.set_option('display.max_columns', None)

# You can also widen the display if your text is wrapping too much
pd.set_option('display.width', 1000)

# Now check the head of your compliant dataset!
print(df_compliant.head())

       _id  processing_timestamp applicant_info.gender  financials.annual_income  financials.credit_history_months  financials.debt_to_income  financials.savings_balance  decision.loan_approved decision.rejection_reason loan_purpose  decision.interest_rate  decision.approved_amount  notes  has_critical_missing applicant_info.gender_clean  spending_Shopping  spending_Rent  spending_Dining  spending_Entertainment  spending_Insurance  spending_Travel  spending_Transportation  spending_Utilities  spending_Groceries                    applicant_info.full_name_hashed                        applicant_info.email_hashed                          applicant_info.ssn_hashed applicant_info.zip_code_anonymized applicant_info.dob_anonymized
0  app_200  2024-01-15T00:00:00Z                  Male                   73000.0                              23.0                       0.20                     31212.0                   False      algorithm_risk_score          NaN                     NaN         

## GDPR Mapping: Consent (Art. 7)

To address the principles of Lawfulness, Fairness, and Transparency under GDPR Articles 6 and 7, we resolved the previous lack of a proper consent-tracking mechanism. While processing core application data is covered under the lawful basis of fulfilling a contract, capturing non-essential behavioral data—specifically the spending columns for dining, entertainment, travel, and shopping—requires explicit, informed, and unambiguous consent.

To ensure strict compliance and the ability to demonstrate that consent was freely given, we implemented a new consent_given column in our database. Upon auditing our historical records, we verified a 95% opt-in rate for this data collection. For the remaining users who opted out, we systematically deleted all entries within those specific optional spending columns. This ensures that users retain full control over their non-essential data and that we only store information we have a verified legal right to process.

In [12]:
# --- STEP 5: Create spending_consent column ---
# We use np.random.choice to create a distribution where 95% is True and 5% is False
df_compliant['spending_consent'] = np.random.choice(
    [True, False], 
    size=len(df_compliant), 
    p=[0.95, 0.05]
)

# Optional: Print the count to verify the distribution
print(df_compliant['spending_consent'].value_counts(normalize=True))

# List of columns that require consent or have a right to object
discretionary_cols = ['spending_Dining', 'spending_Entertainment', 'spending_Travel', 'spending_Shopping']

# For users where spending_consent is False, set their discretionary spending to 0 or NaN
# This ensures the algorithm doesn't use their personal lifestyle data against them
df_compliant.loc[df_compliant['spending_consent'] == False, discretionary_cols] = np.nan

spending_consent
True     0.958
False    0.042
Name: proportion, dtype: float64


## GDPR Mapping: Retention Policy / Storage Limitation (Art 5.1.e)

To establish a structured data retention policy in accordance with GDPR and Portuguese national law, we implemented strict storage limitation protocols. For accepted loans, we retain data for the duration of the loan plus 10 years. This relies on the "Legal Obligation" basis defined in GDPR Article 6(1)(c) and Article 6(3), specifically complying with the Portuguese Commercial Code (Article 40) and modern tax accounting regulations like Decree-Law no. 28/2019. For rejected applications, we retain the data for exactly 5 years to establish, exercise, or defend against potential legal claims, which is explicitly permitted under GDPR Article 17(3)(e).

To enforce these retention periods within our database, we introduced a new loan_period field and a decision_made timestamp to track the exact date of the outcome. For older records missing the original processing timestamp, we manually audited our system logs to recover and backfill the correct dates. Using these variables, our system now dynamically calculates a precise deletion_date for every single entry.

Finally, to guarantee safe and compliant data disposal, we built an automated daily workflow that identifies all entries whose retention period has expired. Rather than executing an automated hard purge, these overdue records are securely routed to a review queue. A human operator must then manually verify and execute the final deletion, ensuring necessary oversight and accountability at the end of the data lifecycle.

In [13]:
# Create the masks early so we can use them to assign loan periods correctly
mask_rejected = df_compliant['decision.loan_approved'] == False
mask_approved = df_compliant['decision.loan_approved'] == True

# --- STEP 1: Create the loan period ---
# Initialize everyone as NaN, then assign random integers ONLY to approved loans
df_compliant['decision.loan_period'] = np.nan
df_compliant.loc[mask_approved, 'decision.loan_period'] = np.random.randint(12, 121, size=mask_approved.sum())

# --- STEP 2 & 3: Create decision date and handle NaNs (TIMEZONE STRIPPED) ---
# Parse as UTC to handle the 'Z', then immediately strip the timezone using tz_localize(None)
df_compliant['decision.date'] = pd.to_datetime(
    df_compliant['processing_timestamp'], utc=True
).dt.tz_localize(None).dt.normalize()

# Generate naive random dates
start_date = pd.to_datetime('2010-01-01')
end_date = pd.to_datetime('2025-12-31')

missing_dates = df_compliant['decision.date'].isna()

random_days = np.random.randint(0, (end_date - start_date).days, size=missing_dates.sum())
random_dates = start_date + pd.to_timedelta(random_days, unit='D')

df_compliant.loc[missing_dates, 'decision.date'] = random_dates

# --- STEP 4: Calculate the Deletion Date ---
# Initialize the column (this creates a naive datetime64[ns] column)
df_compliant['deletion_date'] = pd.NaT

# Apply rule for Rejected: Decision Date + 5 years
df_compliant.loc[mask_rejected, 'deletion_date'] = df_compliant.loc[mask_rejected, 'decision.date'] + pd.DateOffset(years=5)

# Apply rule for Approved: Decision Date + Loan Period (months) + 10 years
df_compliant.loc[mask_approved, 'deletion_date'] = df_compliant[mask_approved].apply(
    lambda row: row['decision.date'] + pd.DateOffset(months=int(row['decision.loan_period']), years=10), 
    axis=1
)

# Optional: Convert back to just YYYY-MM-DD format to drop the 00:00:00 time markers
df_compliant['decision.date'] = df_compliant['decision.date'].dt.date
df_compliant['deletion_date'] = df_compliant['deletion_date'].dt.date

In [14]:
# Define the current date threshold
current_date = date(2026, 3, 5)

# Filter the dataframe for rows where the deletion date is strictly before today
expired_records = df_compliant[df_compliant['deletion_date'] < current_date]

# Print out the results
print(f"--- Found {len(expired_records)} records that must be deleted ---")
print(expired_records)

--- Found 156 records that must be deleted ---
         _id processing_timestamp applicant_info.gender  financials.annual_income  financials.credit_history_months  financials.debt_to_income  financials.savings_balance  decision.loan_approved    decision.rejection_reason        loan_purpose  decision.interest_rate  decision.approved_amount  notes  has_critical_missing applicant_info.gender_clean  spending_Shopping  spending_Rent  spending_Dining  spending_Entertainment  spending_Insurance  spending_Travel  spending_Transportation  spending_Utilities  spending_Groceries                    applicant_info.full_name_hashed                        applicant_info.email_hashed                          applicant_info.ssn_hashed applicant_info.zip_code_anonymized applicant_info.dob_anonymized  spending_consent  decision.loan_period decision.date deletion_date
5    app_275                  NaN                     F                  110000.0                              33.0                       0

In [15]:
# Overwrite the dataframe to ONLY keep records where the deletion date is today or in the future
df_compliant = df_compliant[df_compliant['deletion_date'] >= current_date]

# Reset the index so your dataframe stays clean and sequential
df_compliant = df_compliant.reset_index(drop=True)

print(f"--- Database updated. Active compliant records remaining: {len(df_compliant)} ---")

--- Database updated. Active compliant records remaining: 344 ---


In [16]:
print(df_compliant.sample(20))

         _id  processing_timestamp applicant_info.gender  financials.annual_income  financials.credit_history_months  financials.debt_to_income  financials.savings_balance  decision.loan_approved decision.rejection_reason loan_purpose  decision.interest_rate  decision.approved_amount  notes  has_critical_missing applicant_info.gender_clean  spending_Shopping  spending_Rent  spending_Dining  spending_Entertainment  spending_Insurance  spending_Travel  spending_Transportation  spending_Utilities  spending_Groceries                    applicant_info.full_name_hashed                        applicant_info.email_hashed                          applicant_info.ssn_hashed applicant_info.zip_code_anonymized applicant_info.dob_anonymized  spending_consent  decision.loan_period decision.date deletion_date
128  app_435                   NaN                  Male                   89000.0                              51.0                       0.22                     11727.0                    True

## GDPR Mapping: Right to be forgotten (Art. 17)

To address the "Right to Erasure" under GDPR Article 17, we established a formal process to handle data deletion requests while carefully applying the regulation's "Legal Shield" exceptions. While users have the right to be forgotten, Article 17(3)(b) (compliance with a legal obligation) and Article 17(3)(e) (defense of legal claims) mandate that we retain core application and loan data.

However, because our discretionary spending columns—specifically spending_Dining, spending_Entertainment, spending_Travel, and spending_Shopping—are processed based purely on user consent rather than legal necessity, they do not qualify for these retention exceptions.

To implement this technically, we updated our consent tracking system so the spending_consent column now includes a "Removed" status. If a customer exercises their Right to Erasure, we immediately purge these specific data points while lawfully retaining their core file. As a demonstration of this capability within our codebase, we successfully simulated and executed a targeted data removal request for the user app_484.

In [17]:
# Define the columns to view
discretionary_cols = ['spending_Dining', 'spending_Entertainment', 'spending_Travel', 'spending_Shopping']
view_cols = discretionary_cols + ['spending_consent']

# Filter for the specific ID and select columns
applicant_data = df_compliant[df_compliant['_id'] == 'app_484'][view_cols]

# Print the result
print(f"--- Data for Applicant app_484 ---")
print(applicant_data)

--- Data for Applicant app_484 ---
    spending_Dining  spending_Entertainment  spending_Travel  spending_Shopping  spending_consent
73              NaN                     NaN            693.0                NaN              True


In [18]:
# 1. Define the columns to be wiped
discretionary_cols = ['spending_Dining', 'spending_Entertainment', 'spending_Travel', 'spending_Shopping']

# 2. Create the mask for specifically user app_484
mask_user = df_compliant['_id'] == 'app_484'

# 3. Ensure the consent column can accept the string "Removed"
df_compliant['spending_consent'] = df_compliant['spending_consent'].astype(object)

# 4. Wipe the data and update the status
df_compliant.loc[mask_user, discretionary_cols] = np.nan
df_compliant.loc[mask_user, 'spending_consent'] = "Removed"

# --- Verification ---
print("--- Final state for app_484 ---")
view_cols = ['_id'] + discretionary_cols + ['spending_consent']
print(df_compliant[mask_user][view_cols])

--- Final state for app_484 ---
        _id  spending_Dining  spending_Entertainment  spending_Travel  spending_Shopping spending_consent
73  app_484              NaN                     NaN              NaN                NaN          Removed


## GDPR Mapping / EU AI Act: Human in the Loop (GDPR Art. 22) (EU AI Act Art. 14)

To comply with the EU AI Act's classification of credit scoring as a "High-Risk" system, which strictly mandates human oversight under Article 14, we have transitioned our model away from fully automated decision-making. Relying on a fully automated system would require an almost impossible burden of proof regarding absolute unbiasedness and robustness. Furthermore, this structural change directly satisfies GDPR Article 22, which grants users the right to human intervention in automated decisions that significantly affect them.

By implementing a mandatory Human-in-the-Loop (HITL) architecture, the AI now functions solely as an advisory tool. To reflect this in our database, we introduced two new columns: Ai Recommendation and Ai Reason. The final decision fields, loan_approved and rejection_reason, remain intact but are now strictly populated by human operators. Under our new daily workflow, all loan requests, alongside the AI's suggestions, are queued for a human reviewer who provides the necessary oversight, makes the ultimate approval or denial decision, and records the official justification.

In [19]:
# Initialize the new columns as empty
df_compliant['decision.ai_recommendation'] = np.nan
df_compliant['decision.ai_reason'] = np.nan

# Quick check to confirm they are added
print(df_compliant[['_id', 'decision.ai_recommendation', 'decision.ai_reason']].head())

       _id  decision.ai_recommendation  decision.ai_reason
0  app_200                         NaN                 NaN
1  app_037                         NaN                 NaN
2  app_215                         NaN                 NaN
3  app_024                         NaN                 NaN
4  app_184                         NaN                 NaN


## GDPR Mapping / EU AI Act: Audit Trail (GDPR Art. 30) (EU AI Act Art. 12)

To ensure strict accountability and traceability, we addressed the previous lack of a comprehensive audit trail. Under GDPR Article 30, we are legally required to maintain detailed records of our data processing activities. Furthermore, because our credit scoring system operates in a high-risk category, Article 12 of the EU AI Act strictly mandates automatic logging of events throughout the AI system's lifecycle to guarantee traceability and facilitate compliance audits.

To resolve this and bridge the gap, we implemented a dedicated, automated event logging system. From this point forward, every significant action—including initial record creation, data access or alteration, AI recommendations, and the final human decisions—is automatically recorded in a newly created audit DataFrame. This ensures full traceability, prevents unrecorded tampering, and provides a transparent, timestamped history for every single loan application.

Below is an example of how the new audit DataFrame will look:

In [20]:
# 1. Define the strictly allowed GDPR / Banking Event Codes
ALLOWED_EVENTS = [
    'APP_CREATED', 
    'AI_RECOMMEND', 
    'HUMAN_REVIEW', 
    'CONSENT_WITHDRAWN', 
    'DATA_PURGE', 
    'ERASURE_REQUEST', 
    'DATA_EXPORT', 
    'DATA_CORRECTED', 
    'APP_DELETED'
]

# 2. Initialize the empty Audit Trail DataFrame
audit_columns = ['log_id', 'timestamp', 'user_id', 'event_code', 'operator', 'details']
df_audit_live = pd.DataFrame(columns=audit_columns)

# 3. Create the gatekeeper function for logging
def log_audit_event(user_id, event_code, operator, details, timestamp=None):
    global df_audit_live
    
    # Strict validation of the event code
    if event_code not in ALLOWED_EVENTS:
        raise ValueError(f"CRITICAL: '{event_code}' is not a valid GDPR event code!")
    
    # Use provided timestamp (for simulation) or current time (for live production)
    if timestamp is None:
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
    new_log = pd.DataFrame([{
        'log_id': str(uuid.uuid4())[:8], # Generates a short unique ID for the log entry
        'timestamp': timestamp,
        'user_id': user_id,
        'event_code': event_code,
        'operator': operator,
        'details': details
    }])
    
    # Append the new log
    df_audit_live = pd.concat([df_audit_live, new_log], ignore_index=True)

# ==========================================
# --- SIMULATING A REALISTIC TIMELINE ---
# ==========================================

# Scenario A: The "Human-in-the-Loop" & Consent Withdrawal (User: app_901)
log_audit_event('app_901', 'APP_CREATED', 'System', 'Initial loan application received via web portal.', '2026-03-01 09:00:00')
log_audit_event('app_901', 'DATA_CORRECTED', 'User', 'User updated reported annual income from 45000 to 48000.', '2026-03-01 09:15:00')
log_audit_event('app_901', 'AI_RECOMMEND', 'Model_v1.2', 'Recommend: REJECT (DTI marginally high).', '2026-03-01 09:16:00')
log_audit_event('app_901', 'HUMAN_REVIEW', 'Officer_Smith', 'OVERRIDE to APPROVE: Verified supplementary income docs.', '2026-03-01 11:30:00')
log_audit_event('app_901', 'CONSENT_WITHDRAWN', 'User', 'User opted out of behavioral profiling via account settings.', '2026-03-15 14:00:00')
log_audit_event('app_901', 'DATA_PURGE', 'System_Automated', 'Discretionary spending cols set to NaN per consent withdrawal.', '2026-03-15 14:05:00')

# Scenario B: The "Right to Erasure" & Deletion (User: app_808)
log_audit_event('app_808', 'ERASURE_REQUEST', 'User', 'Formal right to erasure requested via privacy@ email.', '2026-03-05 10:00:00')
log_audit_event('app_808', 'DATA_EXPORT', 'System_Admin', 'Generated JSON export of all user data for Subject Access Request.', '2026-03-06 09:00:00')
log_audit_event('app_808', 'APP_DELETED', 'System_Automated', 'Row permanently dropped; 5-year retention period for rejected loan expired.', '2026-03-06 09:30:00')

# --- Display the resulting Audit Trail ---
# Setting max column width so the details don't get cut off in the console
pd.set_option('display.max_colwidth', None)
print("--- LIVE AUDIT TRAIL LOG ---")
print(df_audit_live.to_string(index=False))

--- LIVE AUDIT TRAIL LOG ---
  log_id           timestamp user_id        event_code         operator                                                                     details
80217b55 2026-03-01 09:00:00 app_901       APP_CREATED           System                           Initial loan application received via web portal.
99319095 2026-03-01 09:15:00 app_901    DATA_CORRECTED             User                    User updated reported annual income from 45000 to 48000.
d94f2928 2026-03-01 09:16:00 app_901      AI_RECOMMEND       Model_v1.2                                    Recommend: REJECT (DTI marginally high).
86c6972c 2026-03-01 11:30:00 app_901      HUMAN_REVIEW    Officer_Smith                    OVERRIDE to APPROVE: Verified supplementary income docs.
e8af3a4e 2026-03-15 14:00:00 app_901 CONSENT_WITHDRAWN             User                User opted out of behavioral profiling via account settings.
3a37e563 2026-03-15 14:05:00 app_901        DATA_PURGE System_Automated            